# Deep Learning StellarNet

Here we build and train StellarNet, a residual MLP designed for tabular data. The goal is to have a neural network in the ensemble alongside the tree-based models. Because its error patterns differ from gradient boosting, it adds diversity to the final stacking step even when it doesn't individually outperform them.

By the end we'll have OOF predictions and test probabilities from the neural network, saved ready for the stacking notebook.

## Imports

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score,classification_report
from src.utils import show_confusion_matrix,plot_roc_curve
from sklearn.preprocessing import StandardScaler
import copy
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

## Data

Same setup as the tuning notebook: load the combined processed data, apply SMOTE to balance the classes, and split into features and target.

In [2]:
test_df_FE = pd.read_csv(r'..\data\processed\test_df_FE.csv')
combined_df_FE = pd.read_csv(r'..\data\processed\combined_df_FE.csv')

X = combined_df_FE.drop(['class'],axis=1)
y = combined_df_FE['class']

target_names =['GALAXY', 'QSO', 'STAR']
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

## Config and device

All training hyperparameters live here so they're easy to adjust in one place. Note that `num_workers` is set to 0 for notebook compatibility increase it when running as a standalone script.

In [ ]:
CONFIG = {
    "batch_size" : 4096,
    "epochs": 120,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "patience": 7,
    "n_splits": 5,
    "random_state": 42,
    "num_workers": 0, # Num workers are not working on notebooks, please increase it when you need to run the NN Train fucntion in py
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Dataset class

`TabularDataset` wraps our numpy arrays in a PyTorch Dataset so the DataLoader can handle batching and shuffling. When `y` is None (the test set case), it just returns features.

In [4]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray = None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

## Model architecture StellarNet

StellarNet is a residual MLP: an input projection followed by a stack of residual blocks, each with two linear layers, BatchNorm, GELU activations, and dropout. Residual connections let gradients flow cleanly through deep networks and help prevent any single block from collapsing.

We use GELU instead of ReLU because it tends to perform better on tabular tasks. The output head is a single linear layer mapping to the 3 class logits.

In [ ]:
class StellarNet(nn.Module):

    def __init__(self, input_dim: int, num_classes: int = 3, hidden_dims: list = [512, 512, 256, 256, 128], dropout: float = 0.25):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.blocks = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            in_d  = hidden_dims[i]
            out_d = hidden_dims[i + 1]
            block = nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(out_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            shortcut = (
                nn.Linear(in_d, out_d, bias=False)
                if in_d != out_d else nn.Identity()
            )
            self.blocks.append(nn.ModuleDict({
                "block": block,
                "shortcut": shortcut,
            }))

        self.head = nn.Linear(hidden_dims[-1], num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        for m in self.blocks:
            residual = m["shortcut"](x)
            x = m["block"](x) + residual
        return self.head(x)

## Training loop

We use Automatic Mixed Precision (AMP) to speed up training on the GPU. Gradient clipping keeps the training stable with the residual architecture.

In [16]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total+= len(y_batch)

    return total_loss / total, total_correct / total

The evaluation loop mirrors the training loop but with gradients disabled.

In [17]:
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    all_preds = []
    all_probs = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
    
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)
        all_preds.extend(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)

    return total_loss / total, total_correct / total, np.array(all_preds), all_probs

## Cross-validation pipeline

`train_pipeline_nn` runs the full 5-fold CV. For each fold it fits a fresh StandardScaler on the training split (the scaler never sees the validation data), trains StellarNet with early stopping on validation loss, loads the best weights before predicting, and stores the out-of-fold probabilities.

In [22]:
def train_pipeline_nn(X: pd.DataFrame, y: pd.Series, config: dict = CONFIG):

    X_arr = X.values.astype(np.float32)
    y_arr = y.values.astype(np.int64)

    input_dim   = X_arr.shape[1]
    num_classes = len(np.unique(y_arr))

    cv = StratifiedKFold(
        n_splits=config["n_splits"],
        shuffle=True,
        random_state=config["random_state"],
    )
    
    n_classes = len(np.unique(y))
    oof_probas = np.zeros((len(y), n_classes))
    oof_preds  = np.zeros(len(y_arr), dtype=np.int64)
    all_scores = []
    all_history = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_arr, y_arr), start=1):
        print(f"\n{'='*55}")
        print(f"Fold {fold} / {config['n_splits']}")
        print("="*55)

        X_tr, X_vl = X_arr[train_idx], X_arr[valid_idx]
        y_tr, y_vl = y_arr[train_idx], y_arr[valid_idx]

        scaler_feat = StandardScaler()
        X_tr = scaler_feat.fit_transform(X_tr)
        X_vl = scaler_feat.transform(X_vl)

        # DataLoaders
        train_ds = TabularDataset(X_tr, y_tr)
        valid_ds = TabularDataset(X_vl, y_vl)

        train_loader = DataLoader(
            train_ds,
            batch_size=config["batch_size"],
            shuffle=True,
            num_workers=config["num_workers"],
            pin_memory=True,)
        
        valid_loader = DataLoader(
            valid_ds,
            batch_size=config["batch_size"] * 2,
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=True,)
        

        # Model
        model = StellarNet(input_dim=input_dim, num_classes=num_classes).to(DEVICE)

        # class_counts = np.bincount(y_tr)
        # class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(DEVICE)
        
        criterion = nn.CrossEntropyLoss()

        # Optimizer + Scheduler
        optimizer = optim.AdamW(
            model.parameters(),
            lr=config["lr"],
            weight_decay=config["weight_decay"],)
        
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=config["epochs"],
            eta_min=1e-6,
        )

        # AMP scaler
        amp_scaler = torch.amp.GradScaler()

        best_val_loss = float("inf")
        best_val_acc= 0.0
        best_weights = None
        patience_count= 0
        history = {"train_loss": [], "train_acc": [],
                    "val_loss": [],   "val_acc": []}

        for epoch in range(1, config["epochs"] + 1):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, amp_scaler)
            vl_loss, vl_acc, _ , _ = eval_epoch(model, valid_loader, criterion)
            scheduler.step()

            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(vl_loss)
            history["val_acc"].append(vl_acc)

            print(f"Epoch {epoch:>3}/{config['epochs']} | Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | Val Loss:   {vl_loss:.4f}  Acc: {vl_acc:.4f}")

            # Early stopping on val_loss
            if vl_loss < best_val_loss:
                best_val_loss= vl_loss
                best_val_acc= vl_acc
                best_weights= copy.deepcopy(model.state_dict())
                patience_count= 0
            else:
                patience_count += 1
                if patience_count >= config["patience"]:
                    print(f"\nEarly stopping at epoch {epoch}")
                    break

        # Load best weights then predict
        model.load_state_dict(best_weights)
        _, _, fold_preds,fold_probas = eval_epoch(model, valid_loader, criterion)
        oof_preds[valid_idx] = fold_preds
        oof_probas[valid_idx] = fold_probas

        fold_score = balanced_accuracy_score(y_vl, fold_preds)
        all_scores.append(fold_score)
        all_history.append(history)

        print(f"\nFold {fold} Best Val Acc: {best_val_acc:.5f}")
        print(f"Fold {fold} Final Score:  {fold_score:.5f}")

    mean_score = np.mean(all_scores)
    print(f"\n{'='*55}")
    print(f"Mean CV Accuracy: {mean_score:.5f}")
    print(f"{'='*55}\n")

    return oof_preds, oof_probas, all_history, mean_score

Run the CV.

In [ ]:
oof_nn_preds,oof_nn_probas, history_nn, cv_score_nn = train_pipeline_nn(X_resampled, y_resampled)

## Final model

`train_final_nn` trains on the full resampled dataset without any validation split. We use the same number of epochs the CV converged to on average, so we're not over-training relative to what the CV validated.

In [ ]:
def train_final_nn(X, y, config):
    
    X_arr = X.values.astype(np.float32)
    y_arr = y.values.astype(np.int64)
    
    scaler_feat = StandardScaler()
    X_arr = scaler_feat.fit_transform(X_arr)
    
    dataset = TabularDataset(X_arr, y_arr)
    loader  = DataLoader(dataset, batch_size=config["batch_size"],
                         shuffle=True, num_workers=config["num_workers"],
                         pin_memory=True)
    
    model = StellarNet(input_dim=X_arr.shape[1], num_classes=3).to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(model.parameters(), lr=config["lr"],
                            weight_decay=config["weight_decay"])
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config["epochs"],
        eta_min=1e-6,
    )
    amp_scaler = torch.amp.GradScaler()
    
    for epoch in range(1, config["epochs"] + 1):
        train_epoch(model, loader, criterion, optimizer, amp_scaler)
        scheduler.step()
    
    return model, scaler_feat

In [ ]:
# scale the test features with the scaler from the final fit, not the CV scalers
final_nn_model, final_scaler = train_final_nn(X_resampled, y_resampled,CONFIG)

X_test_arr = final_scaler.transform(test_df_FE.values.astype(np.float32))
test_dataset = TabularDataset(X_test_arr)
test_loader  = DataLoader(test_dataset, batch_size=CONFIG["batch_size"] * 2,
                          shuffle=False)

final_nn_model.eval()
nn_test_probas = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits  = final_nn_model(X_batch)
        probas  = torch.softmax(logits, dim=1)
        nn_test_probas.append(probas.cpu().numpy())

## Save predictions

Save the OOF predictions, OOF probabilities, and test probabilities to disk for use in the stacking notebook.

In [ ]:
np.save('oof_preds/oof_nn_preds.npy', oof_nn_preds)
np.save('oof_preds/oof_nn_probas.npy', oof_nn_probas)
np.save('oof_preds/nn_test_probas.npy', oof_nn_probas)

## Diagnostics

Let's look at the training curves from the first fold to check for overfitting and verify the early stopping fired at a sensible point.

In [ ]:

h = history_nn[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(h['train_loss'], label='Train'); axes[0].plot(h['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(h['train_acc'], label='Train'); axes[1].plot(h['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
show_confusion_matrix(y_resampled,oof_nn_preds,target_names)

In [ ]:
plot_roc_curve(y_resampled,oof_nn_probas,target_names)

In [ ]:
print(classification_report(y_resampled,oof_nn_preds))